# 06 Replicates Analysis

In [12]:
# Initialize
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun

docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

# Define software to use:
## featureCounts for coutning reads to genomic features
featureCounts() {
    docker_run quay.io/biocontainers/subread:2.1.1--h577a1d6_0 featureCounts "$@"
}
featureCounts -v

# Define software to use:
## samtools for processing SAM/BAM files
samtools() {
    docker_run quay.io/biocontainers/samtools:1.17--h00cdaf9_0 samtools "$@"
}
samtools --version

plotHeatmap 3.5.6
bedtools v2.31.1

featureCounts v2.1.1

samtools 1.17
Using htslib 1.17
Copyright (C) 2023 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             /opt/conda/conda-bld/samtools_1680740641038/_build_env/bin/x86_64-conda-linux-gnu-cc
    CPPFLAGS:       -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /usr/local/include
    CFLAGS:         -Wall -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /usr/local/include -fdebug-prefix-map=/opt/conda/conda-bld/samtools_1680740641038/work=/usr/local/src/conda/samtools-1.17 -fdebug-prefix-map=/usr/local=/usr/local/src/conda-prefix
    LDFLAGS:        -Wl,-O2 -Wl,--sort-common -Wl,--as-needed -Wl,-z,relro -Wl,-z,now -Wl,--disable-new-dtags -Wl,--gc-sections -Wl,--allow-shlib-undefined -Wl,-rpath,/usr/local/lib -Wl,-rpath-link,/usr/local/lib -L/usr/local/lib
    HTSDIR:         
    LIBS:           
  

In [3]:
# First make a associative array to store bigWig and peak locations for each sample
fpath="source_data/bg_corrected_bigWigs/"
op="log2"
rep=R1

plotHeatmap() {
    declare -A bigWigFiles
    for group in shCd19 shRunx3 memory early late terminal; do
        for target in Runx3 Runx1; do
            fn=${group}_${target}_${op}.bigWig
            bigWigFiles[${group}_${target}]="${fpath}${fn}"
        done
    done

    # Compute Matrix
    mkdir -p 06_replicates

    deeptools computeMatrix reference-point \
        -S \
        ${bigWigFiles["shCd19_Runx3"]} \
        ${bigWigFiles["shRunx3_Runx3"]} \
        ${bigWigFiles["shCd19_Runx1"]} \
        ${bigWigFiles["shRunx3_Runx1"]} \
        ${bigWigFiles["memory_Runx3"]} \
        ${bigWigFiles["early_Runx3"]} \
        ${bigWigFiles["late_Runx3"]} \
        ${bigWigFiles["terminal_Runx3"]} \
        ${bigWigFiles["memory_Runx1"]} \
        ${bigWigFiles["early_Runx1"]} \
        ${bigWigFiles["late_Runx1"]} \
        ${bigWigFiles["terminal_Runx1"]} \
        -R \
        "01_peakEDA/cluster1.fullpeak.clean.bed" \
        "01_peakEDA/cluster2.fullpeak.clean.bed" \
        --referencePoint center \
        -b 1500 -a 1500 \
        --numberOfProcessors 48 \
        --sortUsing mean \
        -out "06_replicates/martrix.${rep}.${op}.gz"

    deeptools plotHeatmap \
        -m "06_replicates/martrix.${rep}.${op}.gz" \
        -out "06_replicates/martrix.${rep}.${op}.pdf" \
        --sortUsing mean \
        --colorMap  RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r \
                    RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r \
                    RdYlGn_r RdYlGn_r RdYlGn_r RdYlGn_r \
        --samplesLabel  "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                        "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                        "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
        --regionsLabel "Cluster.1" "Cluster.2"
}

plotHeatmap


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698


In [4]:
op="subtract"
plotHeatmap


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698


In [5]:
fpath="source_data/corrected_background_R2/"
op="log2"
rep=R2
plotHeatmap

In [6]:
op="subtract"
plotHeatmap

In [7]:
plotHeatmap() {

    declare -A bigWigFilesR1
    declare -A bigWigFilesR2

    fpath1="source_data/bg_corrected_bigWigs/"
    fpath2="source_data/corrected_background_R2/"

    for group in shCd19 shRunx3 memory early late terminal; do

        for target in Runx3 Runx1; do

            fn=${group}_${target}_${op}.bigWig

            bigWigFilesR1[${group}_${target}]="${fpath1}${fn}"
            bigWigFilesR2[${group}_${target}]="${fpath2}${fn}"

        done
    done

    # Compute Matrix
    mkdir -p 06_replicates

    deeptools multiBigwigSummary BED-file -b \
        ${bigWigFilesR1["shCd19_Runx3"]} \
        ${bigWigFilesR1["shRunx3_Runx3"]} \
        ${bigWigFilesR1["shCd19_Runx1"]} \
        ${bigWigFilesR1["shRunx3_Runx1"]} \
        ${bigWigFilesR1["memory_Runx3"]} \
        ${bigWigFilesR1["early_Runx3"]} \
        ${bigWigFilesR1["late_Runx3"]} \
        ${bigWigFilesR1["terminal_Runx3"]} \
        ${bigWigFilesR1["memory_Runx1"]} \
        ${bigWigFilesR1["early_Runx1"]} \
        ${bigWigFilesR1["late_Runx1"]} \
        ${bigWigFilesR1["terminal_Runx1"]} \
        ${bigWigFilesR2["shCd19_Runx3"]} \
        ${bigWigFilesR2["shRunx3_Runx3"]} \
        ${bigWigFilesR2["shCd19_Runx1"]} \
        ${bigWigFilesR2["shRunx3_Runx1"]} \
        ${bigWigFilesR2["memory_Runx3"]} \
        ${bigWigFilesR2["early_Runx3"]} \
        ${bigWigFilesR2["late_Runx3"]} \
        ${bigWigFilesR2["terminal_Runx3"]} \
        ${bigWigFilesR2["memory_Runx1"]} \
        ${bigWigFilesR2["early_Runx1"]} \
        ${bigWigFilesR2["late_Runx1"]} \
        ${bigWigFilesR2["terminal_Runx1"]} \
        --BED "01_peakEDA/fullpeak.clean.clustered.named.bed" \
        --outRawCounts 06_replicates/signal.${op}.tab \
        -o 06_replicates/signal.${op}.npz

}

op="log2"
plotHeatmap


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
*Warning*
The resulting bed file does not contain information for the chromosomes that were not common between the bigwig files
Number of bins found: 4172


In [8]:
op="subtract"
plotHeatmap


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
*Warning*
The resulting bed file does not contain information for the chromosomes that were not common between the bigwig files
Number of bins found: 4172


## Feature Counts

In [3]:
awk 'BEGIN{OFS="\t"; print "GeneID","Chr","Start","End","Strand"}
     {name = ($4 != "" && $4 != ".") ? $4 : $1"_"$2"_"$3;
      print name, $1, $2+1, $3, "."}' 01_peakEDA/all.clean.noCluster.bed > 06_replicates/all.clean.noCluster.saf

In [6]:
# Make a string for the sample names to use in the featureCounts command
for rep in R1 R2; do
    for group in shCd19 shRunx3 memory early late terminal; do
        for target in Runx3 Runx1; do
            fpath="source_data/bowtie2/target/markdup"
            fn=${group}_${target}_${rep}.target.markdup.sorted.bam
            echo -n "${fpath}/${fn} "
        done
    done
done > 06_replicates/sample_names.txt

cat 06_replicates/sample_names.txt

source_data/bowtie2/target/markdup/shCd19_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/shCd19_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/shRunx3_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/shRunx3_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/memory_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/memory_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/early_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/early_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/late_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/late_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/terminal_Runx3_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/terminal_Runx1_R1.target.markdup.sorted.bam source_data/bowtie2/target/markdup/shCd19_Runx3_R2.target.markdup.sorted.bam

In [9]:
featureCounts \
    -a 06_replicates/all.clean.noCluster.saf -F SAF \
    -o 06_replicates/peakCounts.mat \
    -p --countReadPairs -B \
    -O -T 8 -Q 20 \
    --donotsort \
    $(cat 06_replicates/sample_names.txt)


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.1.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 24 BAM files                                     ||
||                                                                            ||
||                           shCd19_Runx3_R1.target.markdup.sorted.bam        ||
||                           shCd19_Runx1_R1.target.markdup.sorted.bam        ||
||                           shRunx3_Runx3_R1.target.markdup.sort

In [13]:
for b in $(cat 06_replicates/sample_names.txt); do
    s=$(basename "$b" .target.markdup.sorted.bam)
    n=$(samtools view -c -F 0x904 -f 64 -q 20 "$b")   # primary, properly-paired R1, MAPQ≥20
    echo -e "${s}\t${n}"
done > 06_replicates/libsizes.txt